# Suite RFL — Reflection

A schema is an object of its kind's meta-schema, the schema of its module form: a reference object identified by
itself, which writes itself through `accept` as a module writes it, with its name. `Reflection.of(store)` is a store
whose objects are the schemas a store registers and the named schemas they refer to, by kind, in name order; the store's
data is not read. Predicates match schemas there (mbse-patterns), and compare what they read deeply.

In [ ]:
from mbse.Schemas.Framework import Modules, Plain, Proxies, Reflection as R, Schemas as S, Validators as V
from support import raises, text

Phone = S.OfObject.Builder().name("Phone").properties(text("number")).create()
Listing = S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
    "contact").create()
Contact = S.OfObject.Builder().name("Contact").ref().description("A person").properties(
    text("name"), lambda p: p.name("home").of(Phone).description("Where to call"),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda x: x.as_native(str)).extent(0, 9)))).relations(
    lambda a: a.name("listings").of(Listing).me("contact")).create()
n = S.Form.Data("variable", {"name": "n"})
Word = S.OfNative.Builder().name("Word").type(int).parameters(lambda p: p.name("n").of(lambda t: t.as_native(int))).bits(n).create()
Reach = S.OfUnion.Builder().name("Reach").branches(lambda b: b.name("phone").of(Phone), lambda b: b.name("other").of(Phone)).create()
Both = S.OfIntersection.Builder().name("Both").parts(lambda b: b.name("a").of(Phone), lambda b: b.name("b").of(Phone)).create()
Byte = S.OfApply.Builder().name("Byte").of(Word).arguments(8).create()
Keyed = S.OfIndexed.Builder().name("Keyed").key(lambda t: t.as_native(str)).of(Byte).create()
Pair = S.OfRelation.Builder().name("Pair").links("a", "b").create()  # no uniques
Bare = S.OfApply.Builder().name("Bare").of(Word).create()  # no arguments
Loose = S.OfObject.Builder().name("Loose").parameters(lambda p: p.name("x")).create()  # a parameter of any type
Node = S.OfObject.Builder().name("Node").properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("next").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
Twin = S.OfObject.Builder().name("Phone").create()  # another schema of the same name
Holder = S.OfObject.Builder().name("Holder").properties(
    *[lambda p, k=k: p.name(k.name.lower()).of(k) for k in (Word, Reach, Both, Byte, Keyed, Bare, Loose, Node)],
    lambda p: p.name("twin").of(Twin)).create()


def stocked(*schemas):
    """A store of proxies that registers `schemas`."""
    store = Proxies.OfStore()
    for schema in schemas:
        store.register(schema)
    return store


def plain(value):
    """What a value writes through the visitor protocols, as plain data."""
    if isinstance(value, V.ListRecord):
        return [plain(item) for item in value.values]
    if callable(getattr(value, "accept", None)):
        return {name: plain(v) for name, v in V.properties_of(value).items()}
    return value


def module_form(schema):
    """A schema's definition in a module, with its name."""
    modules = Proxies.OfStore()
    entry = Plain.ToPlain(modules)(S.Module.Schema, Modules.module(modules, [schema]))["objects"]["s0"]["schemas"][0]
    (contents,) = entry["schema"].values()
    return {"name": entry["name"], **contents}

## RFL-01 · A store of schemas: what a store registers and the named schemas they refer to, by kind, in name order

In [ ]:
source = stocked(Contact, Listing)
source.Contact().name("Ada").create()  # data, which is not read
store = R.of(source)
assert list(store.extent("Schemas.Object")) == [Contact, Phone] and list(store.extent("Schemas.Relation")) == [Listing]  # Phone: referred to
assert store.names() == tuple(meta.name for meta in R.META) == (
    "Schemas.Native", "Schemas.Object", "Schemas.Union", "Schemas.Intersection", "Schemas.Indexed", "Schemas.Apply",
    "Schemas.Relation")
assert R.META[1] is S.OfObject.Schema and not S.OfObject.Schema.ref  # the module form's own meta-schemas
assert (Contact.schema_name(), Contact.identity() == id(Contact), Contact.owner()) == ("Schemas.Object", True, None)
kinds = R.of(stocked(Holder, Pair))
assert [[s.name for s in kinds.extent(meta.name)] for meta in R.META] == [
    ["Word"], ["Holder", "Loose", "Node", "Phone", "Phone"], ["Reach"], ["Both"], ["Keyed"], ["Bare", "Byte"], ["Pair"]]
assert kinds.extent("Schemas.Object")[3] is Twin  # names that tie keep the order they are reached in
assert list(R.of(Proxies.OfStore()).extent("Schemas.Object")) == []  # a store's own meta-schemas are not its schemas
with raises(AttributeError, match="no schema registered as 'Contact'"):
    store.extent("Contact")
source.register(Pair)
assert list(store.extent("Schemas.Relation")) == [Listing, Pair]  # read from the registry when asked

## RFL-02 · Read through the visitor protocols as its module form, with its name; named schemas by name

In [ ]:
assert plain(Contact) == module_form(Contact) == {
    "name": "Contact", "properties": [
        {"name": "name", "type": {"native": {"format": "basic", "token": "str"}}},
        {"name": "home", "type": {"named": {"name": "Phone"}}, "description": "Where to call"},
        {"name": "tags", "type": {"indexed": {"item": {"native": {"format": "basic", "token": "str"}},
                                              "extent": {"minimum": 0, "maximum": 9}}}}],
    "adjacencies": [{"name": "listings", "relation": {"named": {"name": "Listing"}}, "me": "contact"}],
    "ref": True, "description": "A person"}
assert all(plain(schema) == module_form(schema) for schema in (Phone, Listing, Word, Reach, Both, Byte, Keyed, Pair, Bare,
                                                                Loose, Node, Holder))
assert "name" not in plain(Contact.properties["name"].type)  # an unnamed schema
assert store.member(Contact, "name") == "Contact" and store.member(Contact, "singleton") is None

## RFL-03 · Validated as their meta-schemas; another format's token; an unsupported token; a store of schemas builds nothing

In [ ]:
assert all(V.Validate(kinds)(meta, schema) == [] for meta in R.META for schema in kinds.extent(meta.name))
Size = S.OfNative.resolve(lambda t: t.name("Size").token("ccpp", "size_t"))
assert list(R.of(stocked(S.OfObject.Builder().name("Sized").properties(lambda p: p.name("n").of(Size)).create())).extent(
    "Schemas.Native")) == [Size] and plain(Size) == {"name": "Size", "format": "ccpp", "token": "size_t"}
with raises(TypeError, match="unsupported native type 'x'"):
    V.properties_of(S.OfNative.Data("x"))
with raises(TypeError, match="schemas are built by their builders, not by a store of schemas"):
    store.builder("Schemas.Object")